# ArtSleuth — Forgery Validation (Colab continuation)

Continues the one-class authentication evaluation from artist #39 onwards.
The first 38 artists were completed on Kaggle. This notebook processes the remaining ~87.

In [ ]:
!pip install -q git+https://github.com/openai/CLIP.git datasets scikit-learn
print('Done.')

In [ ]:
import torch, clip, numpy as np, time, json
from datasets import load_dataset
from collections import Counter
from sklearn.metrics import roc_auc_score

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
dino = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14').eval().to(device)
clip_model, clip_preprocess = clip.load('ViT-L/14', device=device)
clip_model = clip_model.float()

from torchvision import transforms as T
dino_transform = T.Compose([
    T.Resize(252, interpolation=T.InterpolationMode.BICUBIC),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)),
])
print('Models loaded.')

In [ ]:
print('Loading WikiArt...')
ds = load_dataset('huggan/wikiart', split='train')
print(f'Total: {len(ds)}')

artist_names = ds.features['artist'].names
all_artist_labels = ds['artist']

unknown_id = next(i for i, n in enumerate(artist_names) if 'unknown' in n.lower())
artist_counts = Counter(all_artist_labels)
qualified = {aid: cnt for aid, cnt in artist_counts.items() if cnt >= 80 and aid != unknown_id}

artist_indices = {}
for i, aid in enumerate(all_artist_labels):
    if aid in qualified:
        artist_indices.setdefault(aid, []).append(i)

all_qualified_idx = [i for idxs in artist_indices.values() for i in idxs]

# Already completed on Kaggle — skip these
DONE = {
    'vincent-van-gogh', 'nicholas-roerich', 'pierre-auguste-renoir',
    'claude-monet', 'pyotr-konchalovsky', 'camille-pissarro',
    'john-singer-sargent', 'rembrandt', 'marc-chagall', 'pablo-picasso',
    'gustave-dore', 'albrecht-durer', 'boris-kustodiev', 'edgar-degas',
    'paul-cezanne', 'ivan-aivazovsky', 'martiros-saryan', 'eugene-boudin',
    'childe-hassam', 'ilya-repin', 'ivan-shishkin', 'raphael-kirchner',
    'salvador-dali', 'alfred-sisley', 'camille-corot', 'isaac-levitan',
    'henri-matisse', 'maurice-prendergast', 'paul-gauguin',
    'william-merritt-chase', 'amedeo-modigliani', 'james-tissot',
    'ernst-ludwig-kirchner', 'joaquin-sorolla', 'zinaida-serebriakova',
    'konstantin-makovsky', 'henri-de-toulouse-lautrec', 'sam-francis',
}
done_ids = {i for i, n in enumerate(artist_names) if n in DONE}

remaining = {aid: cnt for aid, cnt in qualified.items() if aid not in done_ids}
print(f'Already done: {len(done_ids)}, Remaining: {len(remaining)}')
print(f'Total images in remaining artists: {sum(len(artist_indices[a]) for a in remaining)}')

In [ ]:
@torch.no_grad()
def extract_embeddings(indices):
    dino_embs, clip_embs = [], []
    for start in range(0, len(indices), 32):
        batch_idx = indices[start:start + 32]
        dino_imgs, clip_imgs = [], []
        for idx in batch_idx:
            img = ds[int(idx)]['image']
            if img.mode != 'RGB':
                img = img.convert('RGB')
            try:
                dino_imgs.append(dino_transform(img))
                clip_imgs.append(clip_preprocess(img))
            except Exception:
                continue
        if not dino_imgs:
            continue
        dino_batch = torch.stack(dino_imgs).to(device)
        clip_batch = torch.stack(clip_imgs).to(device)
        with torch.amp.autocast('cuda'):
            d = dino(dino_batch).float().cpu().numpy()
            c = clip_model.encode_image(clip_batch).float().cpu().numpy()
        dino_embs.append(d)
        clip_embs.append(c)
    if not dino_embs:
        return np.zeros((0, 768)), np.zeros((0, 768))
    return np.vstack(dino_embs), np.vstack(clip_embs)

def mahalanobis_scores(ref_feats, test_feats):
    mean = ref_feats.mean(axis=0)
    cov = np.cov(ref_feats, rowvar=False) + np.eye(ref_feats.shape[1]) * 1e-6
    try:
        cov_inv = np.linalg.inv(cov)
    except np.linalg.LinAlgError:
        cov_inv = np.linalg.pinv(cov)
    diffs = test_feats - mean
    return np.sqrt(np.maximum(np.sum(diffs @ cov_inv * diffs, axis=1), 0.0))

print('Ready.')

In [ ]:
rng = np.random.RandomState(42)
results = []
t0 = time.time()

sorted_remaining = sorted(remaining.items(), key=lambda x: -x[1])

for rank, (aid, count) in enumerate(sorted_remaining):
    name = artist_names[aid]
    idxs = np.array(artist_indices[aid])
    rng.shuffle(idxs)
    split = int(0.8 * len(idxs))
    ref_idx, genuine_test_idx = idxs[:split], idxs[split:]
    other_idx = [i for i in all_qualified_idx if all_artist_labels[i] != aid]
    impostor_idx = rng.choice(other_idx, size=min(len(genuine_test_idx), len(other_idx)), replace=False)

    print(f'\n[{rank+1}/{len(remaining)}] {name} — {len(ref_idx)} ref, {len(genuine_test_idx)} gen, {len(impostor_idx)} imp')
    t_a = time.time()
    ref_dino, ref_clip = extract_embeddings(ref_idx.tolist())
    gen_dino, gen_clip = extract_embeddings(genuine_test_idx.tolist())
    imp_dino, imp_clip = extract_embeddings(impostor_idx.tolist())

    if ref_dino.shape[0] < 10 or gen_dino.shape[0] < 5 or imp_dino.shape[0] < 5:
        print('  Skipping — too few embeddings'); continue

    row = {'artist': name, 'n_ref': int(len(ref_idx)), 'n_genuine': int(len(genuine_test_idx)), 'n_impostor': int(len(impostor_idx))}
    for tag, rf, gf, imf in [
        ('dino', ref_dino, gen_dino, imp_dino),
        ('clip', ref_clip, gen_clip, imp_clip),
        ('fused', np.hstack([ref_dino, ref_clip]), np.hstack([gen_dino, gen_clip]), np.hstack([imp_dino, imp_clip])),
    ]:
        gs = mahalanobis_scores(rf, gf)
        ims = mahalanobis_scores(rf, imf)
        labels = np.concatenate([np.zeros(len(gs)), np.ones(len(ims))])
        scores = np.concatenate([gs, ims])
        valid = np.isfinite(scores)
        auc = roc_auc_score(labels[valid], scores[valid]) if valid.sum() >= 10 else float('nan')
        row[f'auc_{tag}'] = round(auc, 4)
        print(f'  {tag:>5s} AUC: {auc:.4f}')
    results.append(row)
    print(f'  {time.time()-t_a:.0f}s | Total: {(time.time()-t0)/60:.1f} min')

print(f'\nDone. {len(results)} artists in {(time.time()-t0)/60:.1f} min')

In [ ]:
import pandas as pd
df = pd.DataFrame(results)
print('\n=== COLAB CONTINUATION RESULTS ===')
for tag, label in [('dino','DINOv2'), ('clip','CLIP'), ('fused','Fused')]:
    col = f'auc_{tag}'
    v = df[col].dropna()
    print(f'{label}: mean={v.mean():.4f}  median={v.median():.4f}  min={v.min():.4f}  max={v.max():.4f}')

print('\nPer-artist:')
print(df.sort_values('auc_fused', ascending=False).to_string(index=False))

# Save for download
with open('colab_forgery_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('\nSaved to colab_forgery_results.json — download this file.')